# Parameter Scan & Optimization
This notebook demonstrates Tier-1 analytical parameter scans and Bayesian optimization.

In [ ]:
import numpy as np
from helicon.fields.biot_savart import Coil
from helicon.optimize.analytical import screen_geometry

## 1. Define parameter ranges

In [ ]:
from helicon.optimize.scan import ParameterRange, generate_scan_points

ranges = [
    ParameterRange(name='coil_current', low=5000.0, high=20000.0, n_points=3),
    ParameterRange(name='coil_radius', low=0.05, high=0.2, n_points=3),
]
scan_points = generate_scan_points(ranges)
print(f'Total scan points: {len(scan_points)}')
for i, pt in enumerate(scan_points[:3]):
    print(f'  Point {i}: {pt}')

## 2. Run analytical screening scan

In [ ]:
results = []
for pt in scan_points:
    coils = [Coil(z=0.0, r=pt['coil_radius'], I=pt['coil_current'])]
    res = screen_geometry(coils, z_min=-0.5, z_max=2.0)
    results.append({
        'current': pt['coil_current'],
        'radius': pt['coil_radius'],
        'mirror_ratio': res.mirror_ratio,
        'thrust_efficiency': res.thrust_efficiency,
    })

print(f'Scanned {len(results)} configurations')
for r in results[:3]:
    print(f"  I={r['current']:.0f} A, r={r['radius']:.3f} m -> "
          f"R_B={r['mirror_ratio']:.2f}, eta={r['thrust_efficiency']:.3f}")

## 3. Pareto front

In [ ]:
from helicon.optimize.pareto import pareto_front

# Build cost array: minimize negative efficiency, minimize negative mirror ratio
costs = np.array([[-r['thrust_efficiency'], -r['mirror_ratio']] for r in results])
front_indices = pareto_front(costs)
print(f'Pareto front has {len(front_indices)} points')
for idx in front_indices:
    r = results[idx]
    print(f"  I={r['current']:.0f} A, r={r['radius']:.3f} m -> "
          f"eta={r['thrust_efficiency']:.3f}, R_B={r['mirror_ratio']:.2f}")

## 4. Sensitivity analysis (Sobol indices)

In [ ]:
from helicon.optimize.sensitivity import compute_sobol

param_names = ['coil_current', 'coil_radius']
X = np.array([[pt['coil_current'], pt['coil_radius']] for pt in scan_points])
Y = np.array([r['thrust_efficiency'] for r in results])

sobol = compute_sobol(X, Y, param_names=param_names)
print('Sobol first-order indices:')
for name, si in zip(param_names, sobol.S1):
    print(f'  {name}: S1 = {si:.3f}')

## 5. Next steps

- Use Tier-2 surrogate models for faster exploration: see `helicon.optimize.surrogate`.
- Run full WarpX simulations for the Pareto-optimal configurations.
- See `04_interpret_detachment.ipynb` for detachment analysis.